# Identifiability analysis: Pas+HR vs cath lab vs all 24 sim waveforms

Compares posterior width per parameter across three observation types:
- **`enc-pashr`** (`exp-v1_enc-pashr_maf5_sims`): Pas waveform + HR — minimal input
- **`enc-cathlab`** (`exp-v1_enc-cathlab_maf5_sims`): Prv, Pra, Pvp, Pap + 5 scalars — full cath lab
- **`enc-allwaves`** (`exp-v1_enc-allwaves_maf5_sims`): all 24 continuous sim waveforms — theoretical ceiling

All flows: MAF5, 128-dim encoder, 1M sims. Evaluated on held-out test sims (true theta known).

| Run | Val NLL |
|-----|---------|
| enc-pashr | +22.41 (early stop ep190) |
| enc-cathlab | −18.54 (early stop ep380) |
| enc-allwaves | −29.48 (early stop ep112) |

In [ ]:
import json, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.stats import pearsonr

try:
    ROOT = Path(globals()['_dh'][0]).parent
    assert (ROOT / 'dataset.py').exists()
except:
    ROOT = Path('/home/sa4604/cv-sbi-sim')
sys.path.insert(0, str(ROOT))

from dataset import (
    load_stats, load_manifest,
    CVDataset, ReducedCVDataset, PasHRDataset,
    PARAM_KEYS, PARAM_KEYS_INFER, WAVE_KEYS_CONT,
    N_PARAMS, N_CONT, T, _HR_IDX,
)
from models import AutoencoderEncoder, ReducedAutoencoderEncoder, PasHREncoder, SurrogateDecoder

SIM_ROOT   = Path('/media/local/SimData/hdf5/cv8/simset_10M_cv8Eed_20260314')
STATS_PATH = ROOT / 'norm_stats.json'

OUT_PASHR    = ROOT / 'outputs/exp-v1_enc-pashr_maf5_sims'
OUT_CATHLAB  = ROOT / 'outputs/exp-v1_enc-cathlab_maf5_sims'
OUT_ALLWAVES = ROOT / 'outputs/exp-v1_enc-allwaves_maf5_sims'
OUT_S1       = ROOT / 'outputs/exp-v1_mlp-surrogate_sims'

CACHE_DIR  = ROOT / 'outputs/identifiability_cache'
CACHE_DIR.mkdir(exist_ok=True)

N_TEST    = 1000
N_SAMPLES = 1000
device    = torch.device('cuda:0')

stats    = load_stats(STATS_PATH)
manifest = load_manifest(SIM_ROOT / 'manifest_test.json')

manifest_train = load_manifest(SIM_ROOT / 'manifest_train.json')
prior_lo  = np.array([manifest_train['config']['pvar_low'][k]  for k in PARAM_KEYS_INFER])
prior_hi  = np.array([manifest_train['config']['pvar_high'][k] for k in PARAM_KEYS_INFER])
prior_std = (prior_hi - prior_lo) / np.sqrt(12)

vlv_idx = WAVE_KEYS_CONT.index('Vlv')
vlv_std = stats['waves']['Vlv']['std']

print('ROOT:', ROOT)
print('device:', device)
print('prior_std range:', prior_std.min().round(4), '–', prior_std.max().round(4))

In [ ]:
import time

def run_inference(label, encoder, flow_net, dataset, cache_path):
    """Run flow on N_TEST sims, return theta_true (N, 24) and samples (N, N_SAMPLES, 24)."""
    if cache_path.exists():
        d = np.load(cache_path)
        print(f'[{label}] loaded from cache')
        return d['theta_true'], d['samples']

    print(f'[{label}] running inference on {N_TEST} test sims...')
    theta_list, samples_list = [], []
    t0 = time.time()
    for i in range(N_TEST):
        theta, x = dataset[i]
        with torch.no_grad():
            z = encoder(x.unsqueeze(0).to(device))
            s = flow_net.sample((N_SAMPLES,), condition=z).squeeze(1).cpu()  # (N_SAMPLES, 24)
        theta_list.append(theta.numpy())
        samples_list.append(s.numpy())
        if i % 200 == 0:
            print(f'  {i}/{N_TEST}  ({time.time()-t0:.0f}s)', end='\r', flush=True)

    theta_true = np.stack(theta_list)   # (N_TEST, 24)
    samples    = np.stack(samples_list) # (N_TEST, N_SAMPLES, 24)
    np.savez(cache_path, theta_true=theta_true, samples=samples)
    print(f'[{label}] done  ({time.time()-t0:.0f}s), saved to {cache_path.name}')
    return theta_true, samples


# ── enc-pashr ─────────────────────────────────────────────────────────────────
ds_pashr = PasHRDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats)
enc_pashr = PasHREncoder(latent_dim=128).to(device)
enc_pashr.load_state_dict(torch.load(OUT_PASHR / 'encoder.pt', map_location=device))
enc_pashr.eval()
flow_pashr = torch.load(OUT_PASHR / 'flow_net.pt', map_location=device, weights_only=False)
flow_pashr.eval()

# backward-compat: try new cache name, fall back to old
_cache_pashr = CACHE_DIR / 'posterior_pashr.npz'
if not _cache_pashr.exists() and (CACHE_DIR / 'posterior_a1.npz').exists():
    _cache_pashr = CACHE_DIR / 'posterior_a1.npz'
theta_true_pashr, samples_pashr = run_inference(
    'enc-pashr', enc_pashr, flow_pashr, ds_pashr, _cache_pashr
)
del enc_pashr, flow_pashr; torch.cuda.empty_cache()

# ── enc-cathlab ───────────────────────────────────────────────────────────────
ds_cathlab = ReducedCVDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats)
enc_cathlab = ReducedAutoencoderEncoder(latent_dim=128).to(device)
enc_cathlab.load_state_dict(torch.load(OUT_CATHLAB / 'encoder.pt', map_location=device))
enc_cathlab.eval()
flow_cathlab = torch.load(OUT_CATHLAB / 'flow_net.pt', map_location=device, weights_only=False)
flow_cathlab.eval()

_cache_cathlab = CACHE_DIR / 'posterior_cathlab.npz'
if not _cache_cathlab.exists() and (CACHE_DIR / 'posterior_b1.npz').exists():
    _cache_cathlab = CACHE_DIR / 'posterior_b1.npz'
theta_true_cathlab, samples_cathlab = run_inference(
    'enc-cathlab', enc_cathlab, flow_cathlab, ds_cathlab, _cache_cathlab
)
del enc_cathlab, flow_cathlab; torch.cuda.empty_cache()

# ── enc-allwaves ──────────────────────────────────────────────────────────────
# CVDataset now returns theta_infer (24-dim, HR excluded) — consistent with above
ds_allwaves = CVDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats)
enc_allwaves = AutoencoderEncoder(latent_dim=128).to(device)
enc_allwaves.load_state_dict(torch.load(OUT_ALLWAVES / 'encoder.pt', map_location=device))
enc_allwaves.eval()
flow_allwaves = torch.load(OUT_ALLWAVES / 'flow_net.pt', map_location=device, weights_only=False)
flow_allwaves.eval()

theta_true_allwaves, samples_allwaves = run_inference(
    'enc-allwaves', enc_allwaves, flow_allwaves, ds_allwaves,
    CACHE_DIR / 'posterior_allwaves.npz'
)
del enc_allwaves, flow_allwaves; torch.cuda.empty_cache()

ds_pashr.close(); ds_cathlab.close(); ds_allwaves.close()
print('Inference complete.')
print(f'theta_true: {theta_true_cathlab.shape}  samples: {samples_cathlab.shape}')

## Posterior width per parameter

**Identifiability ratio** = mean posterior std / prior std.  
- 1.0 → observation carries no information (posterior = prior)  
- 0.0 → parameter perfectly constrained by observation

In [ ]:
post_std_pashr    = samples_pashr.std(axis=1).mean(axis=0)     # (24,)
post_std_cathlab  = samples_cathlab.std(axis=1).mean(axis=0)
post_std_allwaves = samples_allwaves.std(axis=1).mean(axis=0)

ident_pashr    = post_std_pashr    / prior_std
ident_cathlab  = post_std_cathlab  / prior_std
ident_allwaves = post_std_allwaves / prior_std

# Sort by allwaves identifiability ratio (most to least identified)
order = np.argsort(ident_allwaves)
params_sorted = [PARAM_KEYS_INFER[i] for i in order]
ip_s  = ident_pashr[order]
ic_s  = ident_cathlab[order]
ia_s  = ident_allwaves[order]

x = np.arange(len(PARAM_KEYS_INFER))
w = 0.25

fig, ax = plt.subplots(figsize=(17, 6))
ax.bar(x - w,   ip_s, w, label='enc-pashr',    color='steelblue',     alpha=0.85)
ax.bar(x,        ic_s, w, label='enc-cathlab',  color='tomato',        alpha=0.85)
ax.bar(x + w,   ia_s, w, label='enc-allwaves',  color='mediumseagreen', alpha=0.85)
ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--', label='prior (uninformative)')
ax.set_xticks(x); ax.set_xticklabels(params_sorted, rotation=55, ha='right', fontsize=8)
ax.set_ylabel('Posterior std / prior std  (lower = more identified)')
ax.set_title('Identifiability ratio per parameter — enc-pashr vs enc-cathlab vs enc-allwaves', fontsize=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.15)
plt.tight_layout(); plt.show()

print(f'\n{"Parameter":<14} {"Prior std":>10} '
      f'{"pashr ratio":>12} {"cathlab ratio":>14} {"allwaves ratio":>15}')
print('-' * 72)
for i in order:
    name = PARAM_KEYS_INFER[i]
    print(f'{name:<14} {prior_std[i]:>10.4f} '
          f'{ident_pashr[i]:>12.3f} {ident_cathlab[i]:>14.3f} {ident_allwaves[i]:>15.3f}')

## R² and calibration — posterior mean vs true theta

In [ ]:
means_pashr    = samples_pashr.mean(axis=1)    # (N_TEST, 24)
means_cathlab  = samples_cathlab.mean(axis=1)
means_allwaves = samples_allwaves.mean(axis=1)

r2_pashr, r2_cathlab, r2_allwaves = np.zeros(24), np.zeros(24), np.zeros(24)
mape_pashr, mape_cathlab, mape_allwaves = np.zeros(24), np.zeros(24), np.zeros(24)

for i in range(24):
    t = theta_true_cathlab[:, i]
    for r2, mape, means in [
        (r2_pashr,    mape_pashr,    means_pashr),
        (r2_cathlab,  mape_cathlab,  means_cathlab),
        (r2_allwaves, mape_allwaves, means_allwaves),
    ]:
        r2[i]   = pearsonr(t, means[:, i])[0] ** 2
        mape[i] = np.mean(np.abs(means[:, i] - t) / (np.abs(t) + 1e-9)) * 100

# Sorted by allwaves R² descending
order_r2 = np.argsort(-r2_allwaves)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
w = 0.25

ax = axes[0]
x = np.arange(24)
ax.bar(x - w,  r2_pashr[order_r2],    w, label='enc-pashr',    color='steelblue',      alpha=0.85)
ax.bar(x,       r2_cathlab[order_r2],  w, label='enc-cathlab',  color='tomato',         alpha=0.85)
ax.bar(x + w,  r2_allwaves[order_r2], w, label='enc-allwaves', color='mediumseagreen', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels([PARAM_KEYS_INFER[i] for i in order_r2],
                                      rotation=55, ha='right', fontsize=8)
ax.set_ylabel('R² (posterior mean vs true)'); ax.set_ylim(0, 1.05)
ax.set_title('R² per parameter', fontsize=11); ax.legend(fontsize=9)

ax = axes[1]
ax.scatter(r2_cathlab, r2_allwaves, s=30, alpha=0.8, color='mediumseagreen',
           label='allwaves vs cathlab')
for i, name in enumerate(PARAM_KEYS_INFER):
    if abs(r2_allwaves[i] - r2_cathlab[i]) > 0.08 or r2_allwaves[i] > 0.85:
        ax.annotate(name, (r2_cathlab[i], r2_allwaves[i]), fontsize=7,
                    xytext=(4, 2), textcoords='offset points')
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='equal')
ax.set_xlabel('enc-cathlab R²'); ax.set_ylabel('enc-allwaves R²')
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.set_title('enc-cathlab vs enc-allwaves R² — points above diagonal = allwaves adds info', fontsize=10)
ax.legend(fontsize=9)

plt.tight_layout(); plt.show()

print(f'\n{"Parameter":<14} {"pashr R²":>10} {"cathlab R²":>12} {"allwaves R²":>13} '
      f'{"pashr MAPE":>12} {"cathlab MAPE":>14} {"allwaves MAPE":>15}')
print('-' * 94)
for i in order_r2:
    print(f'{PARAM_KEYS_INFER[i]:<14} {r2_pashr[i]:>10.3f} {r2_cathlab[i]:>12.3f} '
          f'{r2_allwaves[i]:>13.3f} {mape_pashr[i]:>11.1f}% {mape_cathlab[i]:>13.1f}% '
          f'{mape_allwaves[i]:>14.1f}%')

## Calibration curves — A1 vs B1 on test sims

In [ ]:
alphas = np.linspace(0.05, 0.95, 19)
cov_pashr    = np.zeros((len(alphas), 24))
cov_cathlab  = np.zeros((len(alphas), 24))
cov_allwaves = np.zeros((len(alphas), 24))

for ai, alpha in enumerate(alphas):
    lo_q, hi_q = (1 - alpha) / 2 * 100, (100 - (1 - alpha) / 2 * 100)
    for cov, samples in [
        (cov_pashr,    samples_pashr),
        (cov_cathlab,  samples_cathlab),
        (cov_allwaves, samples_allwaves),
    ]:
        lo = np.percentile(samples, lo_q, axis=1)
        hi = np.percentile(samples, hi_q, axis=1)
        inside = (theta_true_cathlab >= lo) & (theta_true_cathlab <= hi)
        cov[ai] = inside.mean(axis=0)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
runs = [
    ('enc-pashr',    cov_pashr,    'steelblue'),
    ('enc-cathlab',  cov_cathlab,  'tomato'),
    ('enc-allwaves', cov_allwaves, 'mediumseagreen'),
]
for ax, (label, cov, color) in zip(axes, runs):
    ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='ideal')
    for i in range(24):
        ax.plot(alphas, cov[:, i], color=color, alpha=0.25, linewidth=0.8)
    ax.plot(alphas, cov.mean(axis=1), color=color, linewidth=2.5, label='mean across params')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Nominal coverage'); ax.set_ylabel('Empirical coverage')
    ax.set_title(f'{label} — sim calibration (n={N_TEST})', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('Credible interval calibration on test sims', fontsize=12)
plt.tight_layout(); plt.show()

idx_90 = alphas.searchsorted(0.90)
print(f'Mean 90% CI coverage:')
print(f'  enc-pashr:    {cov_pashr[idx_90].mean():.3f}')
print(f'  enc-cathlab:  {cov_cathlab[idx_90].mean():.3f}')
print(f'  enc-allwaves: {cov_allwaves[idx_90].mean():.3f}')
print(f'  (ideal = 0.90)')

## SV via S1 surrogate

Push posterior theta samples through the surrogate decoder (θ → Vlv waveform), 
then derive SV = (max − min) × vlv_std. Compares how well each observation type 
constrains stroke volume — a clinically important derived quantity.

In [ ]:
# Load surrogate and theta normalisation
with open(OUT_S1 / 'theta_norm.json') as f:
    theta_norm = json.load(f)
theta_norm_mean = torch.tensor(theta_norm['mean'], dtype=torch.float32).to(device)  # (25,)
theta_norm_std  = torch.tensor(theta_norm['std'],  dtype=torch.float32).to(device)

surrogate = SurrogateDecoder(hidden=512, n_layers=4).to(device)
surrogate.load_state_dict(torch.load(OUT_S1 / 'decoder.pt', map_location=device))
surrogate.eval()
print('Surrogate loaded:', surrogate.describe())

# True HR from test sims (needed to insert into 24-dim pashr/cathlab samples)
from dataset import SurrogateDataset
ds_surr = SurrogateDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats)
hr_true = np.array([ds_surr[i][0][_HR_IDX].item() for i in range(N_TEST)])  # (N_TEST,)
ds_surr.close()

SV_CACHE = CACHE_DIR / 'sv_posteriors_3way.npz'

if SV_CACHE.exists():
    d = np.load(SV_CACHE)
    sv_pashr = d['sv_pashr']; sv_cathlab = d['sv_cathlab']
    sv_allwaves = d['sv_allwaves']; sv_true = d['sv_true']
    print('SV posteriors loaded from cache.')
else:
    def derive_sv_24(samples_24, hr_true_arr):
        """samples_24: (N_TEST, N_SAMPLES, 24); HR inserted from hr_true_arr."""
        sv_out = np.zeros((N_TEST, N_SAMPLES))
        for i in range(N_TEST):
            s24 = torch.from_numpy(samples_24[i]).float()         # (N_SAMPLES, 24)
            hr_col = torch.full((N_SAMPLES, 1), hr_true_arr[i])
            s25 = torch.cat([s24[:, :_HR_IDX], hr_col, s24[:, _HR_IDX:]], dim=1)
            s25_z = (s25.to(device) - theta_norm_mean) / theta_norm_std
            with torch.no_grad():
                waves_z = surrogate(s25_z)                         # (N_SAMPLES, N_CONT*T)
            waves_z = waves_z.cpu().numpy().reshape(N_SAMPLES, N_CONT, T)
            vlv_z = waves_z[:, vlv_idx, :]
            sv_out[i] = (vlv_z.max(axis=1) - vlv_z.min(axis=1)) * vlv_std
        return sv_out

    # True SV from test sims
    ds_surr2 = SurrogateDataset(str(SIM_ROOT / 'test'), manifest['index'][:N_TEST], stats)
    sv_true_list = []
    for i in range(N_TEST):
        theta_raw, waves_z_true = ds_surr2[i]
        vlv_z_true = waves_z_true.numpy().reshape(N_CONT, T)[vlv_idx]
        sv_true_list.append((vlv_z_true.max() - vlv_z_true.min()) * vlv_std)
    ds_surr2.close()
    sv_true = np.array(sv_true_list)

    print('Deriving SV from enc-pashr posterior...')
    sv_pashr = derive_sv_24(samples_pashr, hr_true)
    print('Deriving SV from enc-cathlab posterior...')
    sv_cathlab = derive_sv_24(samples_cathlab, hr_true)
    print('Deriving SV from enc-allwaves posterior...')
    sv_allwaves = derive_sv_24(samples_allwaves, hr_true)  # allwaves is also 24-dim (HR excluded)

    np.savez(SV_CACHE, sv_pashr=sv_pashr, sv_cathlab=sv_cathlab,
             sv_allwaves=sv_allwaves, sv_true=sv_true)
    print('SV posteriors computed and cached.')

sv_std_pashr    = sv_pashr.std(axis=1)
sv_std_cathlab  = sv_cathlab.std(axis=1)
sv_std_allwaves = sv_allwaves.std(axis=1)
sv_mean_pashr    = sv_pashr.mean(axis=1)
sv_mean_cathlab  = sv_cathlab.mean(axis=1)
sv_mean_allwaves = sv_allwaves.mean(axis=1)

print(f'\nSV posterior std:  enc-pashr={sv_std_pashr.mean():.2f} ml  '
      f'enc-cathlab={sv_std_cathlab.mean():.2f} ml  enc-allwaves={sv_std_allwaves.mean():.2f} ml')
print(f'SV R²:  enc-pashr={pearsonr(sv_true, sv_mean_pashr)[0]**2:.3f}  '
      f'enc-cathlab={pearsonr(sv_true, sv_mean_cathlab)[0]**2:.3f}  '
      f'enc-allwaves={pearsonr(sv_true, sv_mean_allwaves)[0]**2:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

sv_runs = [
    ('enc-pashr',    sv_mean_pashr,    'steelblue'),
    ('enc-cathlab',  sv_mean_cathlab,  'tomato'),
    ('enc-allwaves', sv_mean_allwaves, 'mediumseagreen'),
]
for ax, (label, sv_mean, color) in zip(axes[:3], sv_runs):
    lo = min(sv_true.min(), sv_mean.min())
    hi = max(sv_true.max(), sv_mean.max())
    ax.scatter(sv_true, sv_mean, s=8, alpha=0.5, color=color)
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.8)
    r2   = pearsonr(sv_true, sv_mean)[0] ** 2
    mape = np.mean(np.abs(sv_mean - sv_true) / (np.abs(sv_true) + 1e-9)) * 100
    ax.set_xlabel('True SV (ml)'); ax.set_ylabel('Posterior mean SV (ml)')
    ax.set_title(f'{label}\nSV R²={r2:.3f}  MAPE={mape:.1f}%', fontsize=10)

ax = axes[3]
sv_stds = [
    (sv_std_pashr,    'steelblue',      'enc-pashr'),
    (sv_std_cathlab,  'tomato',         'enc-cathlab'),
    (sv_std_allwaves, 'mediumseagreen', 'enc-allwaves'),
]
all_max = max(s.max() for s, _, _ in sv_stds)
bins = np.linspace(0, all_max * 1.05, 40)
for sv_std, color, label in sv_stds:
    ax.hist(sv_std, bins=bins, color=color, alpha=0.6, label=f'{label}  mean={sv_std.mean():.1f} ml')
ax.set_xlabel('SV posterior std (ml)'); ax.set_ylabel('Count')
ax.set_title('SV posterior uncertainty distribution', fontsize=10)
ax.legend(fontsize=8)

fig.suptitle('Stroke volume: derived from posterior θ via S1 surrogate (θ → Vlv → SV)', fontsize=12)
plt.tight_layout(); plt.show()

## Summary table

In [ ]:
print(f'{"Parameter":<14} {"pashr":>7} {"R²":>6} {"cathlab":>9} {"R²":>6} {"allwaves":>10} {"R²":>6}  Tag')
print('-' * 82)

for i in order:  # sorted by allwaves identifiability (most identified first)
    name = PARAM_KEYS_INFER[i]
    ip, ic, ia = ident_pashr[i], ident_cathlab[i], ident_allwaves[i]

    if ia < 0.10:
        tag = 'fully identified (all)'
    elif ia < 0.25:
        tag = 'well-identified (all)'
    elif ic < 0.25:
        tag = 'cath lab / allwaves; pashr misses'
    elif ip < 0.25:
        tag = 'pashr best; cath lab misses'
    elif ia < 0.5:
        tag = 'partially identified'
    else:
        tag = 'structurally unidentifiable'

    print(f'{name:<14} {ip:>7.3f} {r2_pashr[i]:>6.3f} {ic:>9.3f} {r2_cathlab[i]:>6.3f} '
          f'{ia:>10.3f} {r2_allwaves[i]:>6.3f}  {tag}')

print(f'\nMean ratio:  pashr={ident_pashr.mean():.3f}  cathlab={ident_cathlab.mean():.3f}  '
      f'allwaves={ident_allwaves.mean():.3f}')
print(f'N well-identified (ratio<0.25):  pashr={( ident_pashr<0.25).sum()}  '
      f'cathlab={(ident_cathlab<0.25).sum()}  allwaves={(ident_allwaves<0.25).sum()}')
print(f'N structurally unidentifiable (allwaves ratio>0.5): {(ident_allwaves>0.5).sum()}')